In [2]:
import os
import re
import pandas as pd
from dotenv import load_dotenv
from opendartreader import OpenDartReader
from bs4 import BeautifulSoup

# API 키 로드 및 DART 객체 생성
load_dotenv()
api_key = os.environ.get('DART_API_KEY')
dart = OpenDartReader(api_key)

print("⏳ 데이터 수집 준비 완료...")

# 테스트용 기업(예: 카카오)의 최근 공시 목록 불러오기
# (카카오 고유번호: 00258801)
corp_code = '00258801'
disclosures = dart.list(corp_code, start='20240101') # 2024년 이후 공시

if not disclosures.empty:
    # 가장 최근 공시의 접수번호(rcept_no)와 보고서명(report_nm) 추출
    target_rcept_no = disclosures['rcept_no'].iloc[0]
    target_report_nm = disclosures['report_nm'].iloc[0]
    
    print(f"\n🎯 [타겟 공시 발견]")
    print(f"보고서명: {target_report_nm}")
    print(f"접수번호: {target_rcept_no}")
    
    # OpenDART API를 통해 공시 원문(XML/HTML) 다운로드
    print("\n📥 DART 서버에서 원문 데이터를 긁어오는 중...")
    raw_xml = dart.document(target_rcept_no)
    
    # 전처리 (Data Cleansing)
    # BeautifulSoup으로 HTML 태그 걷어내기
    soup = BeautifulSoup(raw_xml, 'xml')
    clean_text = soup.get_text(separator=' ', strip=True)
    
    # 정규표현식(Regex)으로 불필요한 연속 공백, 특수문자 일부 제거
    clean_text = re.sub(r'\s+', ' ', clean_text) # 띄어쓰기 여러 개를 하나로 통일
    
    # 결과 확인
    print("\n✨ [전처리 완료된 순수 텍스트 미리보기]")
    print("-" * 50)
    print(clean_text[:500] + "\n... (이하 생략)")
    print("-" * 50)
    print(f"📊 총 텍스트 길이: {len(clean_text)} 글자")

else:
    print("❌ 해당 기간에 공시가 없습니다.")

⏳ 데이터 수집 준비 완료...

🎯 [타겟 공시 발견]
보고서명: 대규모기업집단현황공시[분기별공시(대표회사용)]
접수번호: 20260826000632

📥 DART 서버에서 원문 데이터를 긁어오는 중...

✨ [전처리 완료된 순수 텍스트 미리보기]
--------------------------------------------------
기업집단현황공시(분기-대표회사용) 6.2 카카오 대규모기업집단 현황 공시 기업집단명 : 카 카 오 기업집단 동일인 : 김 범 수 기업집단 대표회사 : (주)카카오 작성회사 : (주)카카오 담당자 : 정 현 진 (수석) , 전화번호(1577-3754) 4. 순환출자 현황[ESG] (2) 국내 계열회사간 순환출자 변동 내역[ESG] : 국내 계열회사간 분기별 순환출자 현황 및 변동내역 : 해당사항 없음. 6. 금융ㆍ보험사 의결권 행사 현황[ESG] (1) 금융ㆍ보험사의 국내계열회사주식에 대한 의결권 행사 현황[ESG] (직전 분기 개시일∼종료일 기준, 단위: 주, %) 소속회사명 피출자회사 현황 출자현황 의결권행사 현황 회사명 상장 여부 전체 주식수 주식수 승인 주식수 지분율 주총일 안건 의결권 행사여부 (주)카카오페이 (주)링키지랩 - 69,090 10,363 - 15.00 - 개최실적 없음 - (주)케이큐브홀딩스 (주)카카오 상장 442,981,070 46,253,222 - 10.4
... (이하 생략)
--------------------------------------------------
📊 총 텍스트 길이: 3669 글자


In [6]:
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# HuggingFace에서 KR-FinBERT 모델 로드
model_name = "snunlp/KR-FinBERT-SC"
print("⏳ 금융 특화 AI 모델을 다운로드하고 불러오는 중입니다...")

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)
print("✅ AI 모델 로드 완료!")

# 앞서 추출한 clean_text의 일부 테스트 (BERT의 512 토큰 한계 방어)
# clean_text 변수가 메모리에 살아있다고 가정
test_text = clean_text[:300] 

# 텍스트를 AI가 이해할 수 있는 숫자(Tensor)로 변환
inputs = tokenizer(test_text, return_tensors="pt", truncation=True, max_length=512)

# 모델 추론 (Inference: 긍정/부정/중립 확률 계산)
with torch.no_grad():
    outputs = model(**inputs)
    logits = outputs.logits
    probabilities = F.softmax(logits, dim=-1)

# 결과 해석 (KR-FinBERT-SC 라벨: 0=부정, 1=중립, 2=긍정)
labels = ['📉 부정 (Negative)', '➖ 중립 (Neutral)', '📈 긍정 (Positive)']
predicted_class = torch.argmax(probabilities, dim=-1).item()

print("-" * 50)
print(f"📝 [분석 대상 텍스트]:\n{test_text}...\n")
print("📊 [AI 감성 분석 예측 확률]")
for i, label in enumerate(labels):
    print(f"{label}: {probabilities[0][i].item() * 100:.2f}%")
    
print("-" * 50)
print(f"💡 AI 최종 판별 결과: {labels[predicted_class]}")

⏳ 금융 특화 AI 모델을 다운로드하고 불러오는 중입니다...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

✅ AI 모델 로드 완료!
--------------------------------------------------
📝 [분석 대상 텍스트]:
기업집단현황공시(분기-대표회사용) 6.2 카카오 대규모기업집단 현황 공시 기업집단명 : 카 카 오 기업집단 동일인 : 김 범 수 기업집단 대표회사 : (주)카카오 작성회사 : (주)카카오 담당자 : 정 현 진 (수석) , 전화번호(1577-3754) 4. 순환출자 현황[ESG] (2) 국내 계열회사간 순환출자 변동 내역[ESG] : 국내 계열회사간 분기별 순환출자 현황 및 변동내역 : 해당사항 없음. 6. 금융ㆍ보험사 의결권 행사 현황[ESG] (1) 금융ㆍ보험사의 국내계열회사주식에 대한 의결권 행사 현황[ESG] (직전 분기 개시...

📊 [AI 감성 분석 예측 확률]
📉 부정 (Negative): 0.01%
➖ 중립 (Neutral): 99.99%
📈 긍정 (Positive): 0.01%
--------------------------------------------------
💡 AI 최종 판별 결과: ➖ 중립 (Neutral)


In [10]:
# 텍스트 분할 (Chunking) 설정
chunk_size = 500  # 500글자씩 조각
chunks = [clean_text[i:i+chunk_size] for i in range(0, len(clean_text), chunk_size)]

print(f"전체 텍스트를 {len(chunks)}개의 조각(Chunk)으로 분할했습니다.\n")
print("⏳ 각 조각별로 AI 감성 분석을 시작합니다...\n")

max_negative_prob = 0.0
most_risky_chunk = ""

# 각 조각을 AI 모델에 통과시키기
for i, chunk in enumerate(chunks):
    inputs = tokenizer(chunk, return_tensors="pt", truncation=True, max_length=512)
    
    with torch.no_grad():
        outputs = model(**inputs)
        probabilities = F.softmax(outputs.logits, dim=-1)
        
        # 0번 인덱스가 '부정(Negative)', 1번이 중립, 2번이 긍정
        negative_prob = probabilities[0][0].item() * 100
        
        # 가장 높은 부정 확률 갱신 및 해당 텍스트 저장
        if negative_prob > max_negative_prob:
            max_negative_prob = negative_prob
            most_risky_chunk = chunk
            

# 전체 문서 기준 최종 결과 출력
print("-" * 50)
print(f"📊 [전체 문서 AI 분석 최종 결과]")
print(f"최대 부정(Risk) 확률: {max_negative_prob:.2f}%")

if max_negative_prob >= 50.0:
    print("🚨 [위험 감지] 이 공시에는 부정적인 내용이 다수 포함되어 있습니다.")
    print(f"\n📝 [가장 위험한 부분 발췌]:\n{most_risky_chunk[:200]}...")
else:
    print("✅ [안전] 이 공시에서 특별한 악재성 문맥이 발견되지 않았습니다.")
print("-" * 50)

전체 텍스트를 8개의 조각(Chunk)으로 분할했습니다.

⏳ 각 조각별로 AI 감성 분석을 시작합니다...

--------------------------------------------------
📊 [전체 문서 AI 분석 최종 결과]
최대 부정(Risk) 확률: 99.95%
🚨 [위험 감지] 이 공시에는 부정적인 내용이 다수 포함되어 있습니다.

📝 [가장 위험한 부분 발췌]:
 대법원에서 상고 기각 결정하였습니다. 이에 (주)케이큐브홀딩스는 (주)카카오 및 (주)카카오게임즈에 대하여 의결권 행사가 가능합니다. 7. 계열회사와 특수관계인간 거래현황 (6) 계열회사간 주요 상품ㆍ용역거래 내역 : 국내 계열회사와 상품ㆍ용역을 거래한 금액이 일정 규모 이상인 경우, 그 거래 내역 가. 상장회사와 그 계열회사간 주요 상품ㆍ용역거래 내역 ...
--------------------------------------------------
